In [ ]:
# Step 1: Import Libraries

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score
)

print("Libraries imported successfully!")

Libraries imported successfully!


In [ ]:
# Step 2: Load Dataset

df = pd.read_csv("Sales_Dataset_for_Linear.csv")

print("Dataset loaded successfully!")
print("\nFirst 5 rows:")
print(df.head())

Dataset loaded successfully!

First 5 rows:
      _   Sale_Date Sales_Rep Region  Sales_Amount  Quantity_Sold  \
0  1052  03-02-2023       Bob  North       5053.97             18   
1  1093  21-04-2023       Bob   West       4384.02             17   
2  1015  21-09-2023     David  South       4631.23             30   
3  1072  24-08-2023       Bob  South       2167.94             39   
4  1061  24-03-2023   Charlie   East       3750.20             13   

  Product_Category  Unit_Cost  Unit_Price Customer_Type  Discount  \
0        Furniture     152.75      267.22     Returning      0.09   
1        Furniture    3816.39     4209.44     Returning      0.11   
2             Food     261.56      371.40     Returning      0.20   
3         Clothing    4330.03     4467.75           New      0.02   
4      Electronics     637.37      692.71           New      0.08   

  Payment_Method Sales_Channel Region_and_Sales_Rep  
0           Cash        Online            North-Bob  
1           Cash  

In [ ]:
# Step 3: Explore Dataset

print("Dataset Shape:", df.shape)

print("\nColumn Names:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:", df.duplicated().sum())

print("\nDataset Information:")
df.info()

Dataset Shape: (1000, 14)

Column Names:
['_', 'Sale_Date', 'Sales_Rep', 'Region', 'Sales_Amount', 'Quantity_Sold', 'Product_Category', 'Unit_Cost', 'Unit_Price', 'Customer_Type', 'Discount', 'Payment_Method', 'Sales_Channel', 'Region_and_Sales_Rep']

Data Types:
_                         int64
Sale_Date                object
Sales_Rep                object
Region                   object
Sales_Amount            float64
Quantity_Sold             int64
Product_Category         object
Unit_Cost               float64
Unit_Price              float64
Customer_Type            object
Discount                float64
Payment_Method           object
Sales_Channel            object
Region_and_Sales_Rep     object
dtype: object

Missing Values:
_                       0
Sale_Date               0
Sales_Rep               0
Region                  0
Sales_Amount            0
Quantity_Sold           0
Product_Category        0
Unit_Cost               0
Unit_Price              0
Customer_Type          

In [ ]:
# Step 4: Create Binary Target Variable

median_sales = df["Sales_Amount"].median()

df["High_Sales"] = (
    df["Sales_Amount"] >= median_sales
).astype(int)

print("Median Sales Amount:", median_sales)

print("\nHigh Sales and Low Sales Counts:")
print(df["High_Sales"].value_counts().sort_index())

print("\nTarget Variable:")
print("1 = High Sales")
print("0 = Low Sales")

Median Sales Amount: 5019.3

High Sales and Low Sales Counts:
High_Sales
0    500
1    500
Name: count, dtype: int64

Target Variable:
1 = High Sales
0 = Low Sales


In [ ]:
# Step 5: Select Features and Target

numerical_features = [
    "Quantity_Sold",
    "Unit_Cost",
    "Unit_Price",
    "Discount"
]

categorical_features = [
    "Region",
    "Product_Category",
    "Customer_Type",
    "Sales_Channel"
]

X = df[numerical_features + categorical_features]
y = df["High_Sales"]

print("Input Features (X):")
print(X.head())

print("\nTarget Variable (y):")
print(y.head())

print("\nInput Shape:", X.shape)
print("Target Shape:", y.shape)

Input Features (X):
   Quantity_Sold  Unit_Cost  Unit_Price  Discount Region Product_Category  \
0             18     152.75      267.22      0.09  North        Furniture   
1             17    3816.39     4209.44      0.11   West        Furniture   
2             30     261.56      371.40      0.20  South             Food   
3             39    4330.03     4467.75      0.02  South         Clothing   
4             13     637.37      692.71      0.08   East      Electronics   

  Customer_Type Sales_Channel  
0     Returning        Online  
1     Returning        Retail  
2     Returning        Retail  
3           New        Retail  
4           New        Online  

Target Variable (y):
0    1
1    0
2    0
3    0
4    0
Name: High_Sales, dtype: int64

Input Shape: (1000, 8)
Target Shape: (1000,)


In [ ]:
# Step 6: Preprocess the Features

numerical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numerical_transformer, numerical_features),
    ("cat", categorical_transformer, categorical_features)
])

print("Preprocessing pipeline created successfully!")

Preprocessing pipeline created successfully!


In [ ]:
# Step 7: Split Dataset

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

print("Training Features:", X_train.shape)
print("Testing Features:", X_test.shape)

print("\nTraining Target:", y_train.shape)
print("Testing Target:", y_test.shape)

Training Features: (700, 8)
Testing Features: (300, 8)

Training Target: (700,)
Testing Target: (300,)


In [ ]:
# Step 8: Train Logistic Regression Model

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)

print("Logistic Regression Model Trained Successfully!")

Logistic Regression Model Trained Successfully!


In [ ]:
# Step 9: Make Predictions

y_pred = model.predict(X_test)

print("Actual Values:")
print(y_test.to_numpy())

print("\nPredicted Values:")
print(y_pred)

print("\nFirst 10 Predictions:")

results = pd.DataFrame({
    "Actual": y_test.to_numpy(),
    "Predicted": y_pred
})

print(results.head(10))

Actual Values:
[1 1 1 1 0 1 1 1 1 0 1 0 0 1 1 1 0 0 0 1 0 1 0 0 0 0 1 1 0 1 1 0 1 1 0 1 0
 0 0 1 1 0 1 1 1 0 0 1 1 1 1 0 0 0 0 1 0 0 0 0 0 0 0 0 1 0 0 0 1 1 1 1 0 1
 0 0 1 0 1 1 1 0 0 0 0 0 0 1 1 1 1 1 0 0 1 0 1 0 0 0 1 1 0 1 1 1 1 1 1 0 0
 0 0 0 0 0 0 0 0 0 1 0 1 0 0 1 1 0 1 0 0 1 1 0 0 0 1 0 1 1 1 0 1 1 1 1 0 1
 1 0 0 1 1 0 1 1 1 0 1 1 1 0 0 1 1 0 0 0 0 0 0 0 1 1 0 1 1 0 0 1 0 0 1 1 1
 0 0 1 1 1 1 1 0 0 0 1 1 0 0 1 0 1 1 1 0 1 0 1 0 0 0 1 0 0 0 1 0 1 1 1 1 0
 0 1 0 1 0 1 0 1 1 1 0 1 1 0 0 1 1 1 1 1 0 0 1 0 1 0 1 1 1 1 0 1 1 0 1 1 1
 0 0 0 0 0 1 1 0 1 1 0 1 1 0 1 1 0 0 1 1 0 0 0 1 0 0 1 1 1 0 0 0 1 1 0 0 0
 0 0 0 0]

Predicted Values:
[0 1 0 1 1 1 0 0 0 0 1 0 0 0 0 1 1 0 1 0 1 1 0 0 0 1 1 0 0 1 0 1 1 1 1 1 0
 1 1 0 0 0 0 1 1 0 0 0 1 1 0 1 0 1 1 0 0 0 1 0 0 0 0 0 0 0 1 0 0 1 1 0 0 0
 0 0 1 1 0 1 1 1 0 0 0 1 0 1 0 1 0 1 1 0 1 1 0 1 1 1 0 1 0 1 1 0 0 0 1 1 0
 1 1 1 1 1 0 1 0 0 0 1 0 0 1 1 1 0 0 0 1 1 0 0 0 1 0 1 1 0 0 1 1 1 0 1 1 0
 1 0 1 1 0 0 0 0 1 1 0 0 0 1 1 0 1 1 1 0 1 1 0 0 1 1 0 0

In [ ]:
# Step 10: Evaluate Model Performance

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

print("Model Evaluation Results")
print("------------------------")

print("Accuracy:", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall:", round(recall, 4))
print("F1-Score:", round(f1, 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["Low Sales", "High Sales"],
    zero_division=0
))

Model Evaluation Results
------------------------
Accuracy: 0.4933
Precision: 0.4932
Recall: 0.48
F1-Score: 0.4865

Confusion Matrix:
[[76 74]
 [78 72]]

Classification Report:
              precision    recall  f1-score   support

   Low Sales       0.49      0.51      0.50       150
  High Sales       0.49      0.48      0.49       150

    accuracy                           0.49       300
   macro avg       0.49      0.49      0.49       300
weighted avg       0.49      0.49      0.49       300



In [ ]:
# Step 11: Feature Importance and Conclusion

feature_names = model.named_steps[
    "preprocessor"
].get_feature_names_out()

coefficients = model.named_steps[
    "classifier"
].coef_[0]

feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients,
    "Absolute_Importance": np.abs(coefficients)
})

feature_importance = feature_importance.sort_values(
    by="Absolute_Importance",
    ascending=False
)

print("Top 10 Important Features:")
print(feature_importance.head(10).to_string(index=False))

print("\nFinal Conclusion")
print("----------------")

print(
    f"The Logistic Regression model achieved "
    f"{accuracy * 100:.2f}% accuracy."
)

print(
    "The model predicts whether a sales transaction "
    "belongs to the High Sales or Low Sales category."
)

print(
    "Precision, recall, F1-score and the confusion matrix "
    "help evaluate the model's classification performance."
)

Top 10 Important Features:
                          Feature  Coefficient  Absolute_Importance
                  num__Unit_Price     0.496199             0.496199
                   num__Unit_Cost    -0.405707             0.405707
       cat__Product_Category_Food     0.353506             0.353506
  cat__Product_Category_Furniture    -0.172541             0.172541
                cat__Region_North     0.163151             0.163151
                cat__Region_South    -0.144594             0.144594
   cat__Product_Category_Clothing    -0.104766             0.104766
               num__Quantity_Sold    -0.097467             0.097467
cat__Product_Category_Electronics    -0.073417             0.073417
        cat__Sales_Channel_Retail     0.026951             0.026951

Final Conclusion
----------------
The Logistic Regression model achieved 49.33% accuracy.
The model predicts whether a sales transaction belongs to the High Sales or Low Sales category.
Precision, recall, F1-score and the co